# J-lens replication on Google Colab (free T4)

One-click replication of Anthropic's Jacobian lens on **Qwen2.5-3B-Instruct** using Colab's free T4 GPU.

**Setup:**
1. `Runtime` → `Change runtime type` → **T4 GPU** → Save
2. Optional: `Runtime` → `Manage sessions` → verify you're on T4
3. `Runtime` → `Run all`

**Outputs (saved to Colab session storage):**
- `qwen2.5-3b-jlens.pt` — the fitted Jacobian lens (~40 MB)
- `eval_results.json` — pass@{1,5,10} on the 6 lens-quality evaluations
- `probe_readouts.json` — 3 sanity-check readouts

**Expected runtime:** ~25-40 min on T4. Colab free tier caps at 12h/session so this comfortably fits.

In [ ]:
# Colab's preinstalled torch already supports T4 (sm_75). Just install jlens + deps.
!pip -q install git+https://github.com/anthropics/jacobian-lens.git datasets safetensors
!pip -q install -U 'transformers>=5.5'
!git clone -q --depth=1 https://github.com/anthropics/jacobian-lens.git /content/jl_data

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU! Set Runtime → Change runtime type → T4 GPU'
print(torch.cuda.get_device_name(0), '| compute cap:', torch.cuda.get_device_capability(0))
print('torch:', torch.__version__)

In [ ]:
import pathlib, json, time
import torch, transformers, jlens
from datasets import load_dataset

MODEL = 'Qwen/Qwen2.5-3B-Instruct'    # 3B, Apache 2.0, fits comfortably on T4
N_PROMPTS = 25
MAX_SEQ_LEN = 128
SKIP_FIRST = 4
TARGET_LAYER = -2
DIM_BATCH = 4
OUT_DIR = pathlib.Path('/content')
OUT_LENS = OUT_DIR / 'qwen2.5-3b-jlens.pt'

In [ ]:
hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, attn_implementation='sdpa',
    low_cpu_mem_usage=True,
).to('cuda')
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL)
model = jlens.from_hf(hf_model, tokenizer, compile=True)
print(f'Loaded {MODEL}: n_layers={model.n_layers}  d_model={model.d_model}')

In [ ]:
ds = load_dataset('NeelNanda/pile-10k', split='train')
prompts = []
for row in ds:
    if isinstance(row['text'], str) and len(row['text']) > 200:
        prompts.append(row['text'])
        if len(prompts) >= N_PROMPTS: break
print(f'Selected {len(prompts)} prompts')

In [ ]:
t0 = time.time()
lens = jlens.fit(
    model, prompts=prompts,
    target_layer=TARGET_LAYER, dim_batch=DIM_BATCH,
    max_seq_len=MAX_SEQ_LEN, skip_first=SKIP_FIRST,
    checkpoint_path=str(OUT_DIR / 'ckpt.pt'),
)
print(f'Fit done in {(time.time()-t0)/60:.1f} min')
print(lens)
lens.save(str(OUT_LENS))
print('Peak VRAM:', torch.cuda.max_memory_allocated()/1e9, 'GB')

In [ ]:
probes = [
    'The number of legs on the animal that spins webs is',
    'The capital of the country shaped like a boot is',
    'Fact: The currency used in the country whose flag has a red maple leaf is the',
]
probe_out = {}
for prompt in probes:
    lens_logits, _, _ = lens.apply(model, prompt, positions=[-2])
    per_layer = {}
    for layer in sorted(lens_logits):
        top = lens_logits[layer][0].topk(5).indices.tolist()
        per_layer[layer] = [tokenizer.decode([t]).strip() for t in top]
    probe_out[prompt] = per_layer
    top_final = [tokenizer.decode([t]).strip() for t in lens_logits[max(lens_logits)][0].topk(5).indices.tolist()]
    print(f'{prompt!r}\n  final-layer top-5: {top_final}\n')
with open(OUT_DIR / 'probe_readouts.json', 'w') as f:
    json.dump(probe_out, f, indent=2)

In [ ]:
EVALS_DIR = pathlib.Path('/content/jl_data/data/evaluations')
K_VALUES = (1, 5, 10)

def tokens_of(word):
    ids = []
    for v in {word, ' '+word, word.lower(), ' '+word.lower()}:
        e = tokenizer(v, add_special_tokens=False).input_ids
        if len(e) == 1: ids.append(e[0])
    return sorted(set(ids))

def min_rank_across_layers(lens_logits, token_ids):
    best = 10**9
    for _, logits in lens_logits.items():
        order = logits[0].argsort(descending=True)
        rank_lookup = torch.empty_like(order)
        rank_lookup[order] = torch.arange(order.numel(), device=order.device)
        cand = rank_lookup[torch.tensor(token_ids, device=order.device)]
        r = int(cand.min().item()) + 1
        if r < best: best = r
    return best

eval_results = []
for path in sorted(EVALS_DIR.glob('lens-eval-*.json')):
    data = json.load(path.open())
    items = data['items']
    per_item_frac = {k: [] for k in K_VALUES}
    for item in items:
        try:
            lens_logits, _, _ = lens.apply(model, item['prompt'], positions=[-1])
        except Exception:
            continue
        inters = item['intermediates'] if isinstance(item['intermediates'], list) else [item['intermediates']]
        hits = {k: 0 for k in K_VALUES}
        total = 0
        for inter in inters:
            key = inter if isinstance(inter, str) else (inter.get('synonyms', [inter.get('key','')])[0] if isinstance(inter, dict) else inter[0])
            tok_ids = tokens_of(str(key))
            if not tok_ids: continue
            r = min_rank_across_layers(lens_logits, tok_ids)
            for k in K_VALUES:
                if r <= k: hits[k] += 1
            total += 1
        if total:
            for k in K_VALUES: per_item_frac[k].append(hits[k]/total)
    result = {
        'eval': path.stem,
        'n_items_scored': len(per_item_frac[1]),
        'pass_at_k': {f'pass@{k}': (sum(per_item_frac[k])/max(1,len(per_item_frac[k]))) for k in K_VALUES},
    }
    eval_results.append(result)
    print(path.name, '→', result['pass_at_k'])

with open(OUT_DIR / 'eval_results.json', 'w') as f:
    json.dump({'model': MODEL, 'n_prompts': N_PROMPTS, 'results': eval_results}, f, indent=2)

In [ ]:
# Optional: download results to your machine
from google.colab import files
files.download(str(OUT_LENS))
files.download(str(OUT_DIR / 'eval_results.json'))
files.download(str(OUT_DIR / 'probe_readouts.json'))